## 4. Dijkstra's Algorithm — Implementation from Scratch

In this section i implement Dijkstra's algorithm **without any external libraries**.
i then run it on both mountain graphs with all three cost functions.

---

### 4.1 Why implement from scratch?

Libraries like `networkx` offer `shortest_path()` in one line.
But implementing Dijkstra manually forces us to understand:

- How a **priority queue (min-heap)** works
- Why **greedy** local decisions lead to a globally optimal solution
- The role of the **relaxation step** — the core of the algorithm
- How to **reconstruct the full path** from the `prev[]` array

#### The Relaxation Step

The key operation in Dijkstra is *edge relaxation*. For edge $(u, v)$ with weight $w$:

$$\text{if } dist[u] + w(u,v) < dist[v] \text{ then:}$$
$$dist[v] \leftarrow dist[u] + w(u,v)$$
$$prev[v] \leftarrow u$$

We repeat this for every node extracted from the priority queue, in order of increasing distance.

---

In [ ]:

import heapq   # Python's built-in min-heap
import math


def dijkstra(graph, source, weight_index):
    """
    Dijkstra's shortest path algorithm using a min-heap.

    Parameters:
        graph        : adjacency list — {node: [(neighbor, dist, elev, naismith), ...]}
        source       : starting node index
        weight_index : which weight to optimize
                       1 = distance (meters)
                       2 = elevation gain (meters)
                       3 = Naismith time (hours)

    Returns:
        dist : dict {node: minimum cost from source}
        prev : dict {node: previous node on optimal path}
    """
    # Step 1 — Initialize all distances to infinity
    dist = {node: math.inf for node in graph}
    prev = {node: None    for node in graph}
    dist[source] = 0.0

    # Step 2 — Priority queue: (cost, node)
    # heapq is a min-heap — always pops the smallest element
    pq = [(0.0, source)]

    visited = set()

    # Step 3 — Main loop
    while pq:
        current_cost, u = heapq.heappop(pq)

        # Skip if already processed
        if u in visited:
            continue
        visited.add(u)

        # Step 4 — Relax all edges from u
        for edge in graph[u]:
            v        = edge[0]
            w        = edge[weight_index]   # pick distance, elev, or naismith
            candidate = current_cost + w

            # Relaxation step
            if candidate < dist[v]:
                dist[v] = candidate
                prev[v] = u
                heapq.heappush(pq, (candidate, v))

    return dist, prev


def reconstruct_path(prev, source, target):
    """
    Reconstruct the optimal path from source to target
    using the prev[] dictionary returned by Dijkstra.

    Returns:
        List of node indices from source to target.
        Empty list if no path exists.
    """
    path = []
    node = target
    while node is not None:
        path.append(node)
        node = prev[node]
    path.reverse()

    if path[0] == source:
        return path
    return []   # no path found


print('Dijkstra implementation ready!')
print()
print('Weight index mapping:')
print('  1 -> Distance    (meters)')
print('  2 -> Elev gain   (meters)')
print('  3 -> Naismith    (hours)')

In [ ]:
# ============================================================
# Section 4.2  Run Dijkstra on the Rila Track
# ============================================================
# (Assumes parse_gpx, haversine, build_graph are already defined
#  in Section 3  run that section first!)

import time

SOURCE = 0
TARGET_RILA   = len(rila_nodes)   - 1
TARGET_VIHREN = len(vihren_nodes) - 1

weight_labels = {
    1: ('Shortest distance',  'meters', 1/1000, 'km',    2),
    2: ('Least elevation',    'meters', 1,      'm',     0),
    3: ('Fastest (Naismith)', 'hours',  1,      'hours', 2),
}

print('Running Dijkstra on Rila track...')
print('=' * 50)

rila_results = {}

for w_idx, (label, unit, scale, unit_out, dec) in weight_labels.items():
    t0 = time.time()
    dist_map, prev_map = dijkstra(rila_graph, SOURCE, w_idx)
    elapsed = time.time() - t0

    path     = reconstruct_path(prev_map, SOURCE, TARGET_RILA)
    opt_cost = dist_map[TARGET_RILA] * scale

    rila_results[w_idx] = {
        'label'  : label,
        'path'   : path,
        'cost'   : dist_map[TARGET_RILA],
        'elapsed': elapsed,
    }

    print(f'  {label}')
    print(f'    Optimal cost : {opt_cost:.{dec}f} {unit_out}')
    print(f'    Path length  : {len(path)} nodes')
    print(f'    Runtime      : {elapsed*1000:.1f} ms')
    print()

In [ ]:
# ============================================================
# Section 4.3  Run Dijkstra on the Vihren Track
# ============================================================

print('Running Dijkstra on Vihren track...')
print('=' * 50)

vihren_results = {}

for w_idx, (label, unit, scale, unit_out, dec) in weight_labels.items():
    t0 = time.time()
    dist_map, prev_map = dijkstra(vihren_graph, SOURCE, w_idx)
    elapsed = time.time() - t0

    path     = reconstruct_path(prev_map, SOURCE, TARGET_VIHREN)
    opt_cost = dist_map[TARGET_VIHREN] * scale

    vihren_results[w_idx] = {
        'label'  : label,
        'path'   : path,
        'cost'   : dist_map[TARGET_VIHREN],
        'elapsed': elapsed,
    }

    print(f'  {label}')
    print(f'    Optimal cost : {opt_cost:.{dec}f} {unit_out}')
    print(f'    Path length  : {len(path)} nodes')
    print(f'    Runtime      : {elapsed*1000:.1f} ms')
    print()

In [ ]:
# ============================================================
# Section 4.4 — Validation against networkx
# ============================================================
# We verify our implementation by comparing with networkx's
# built-in shortest_path — results must match exactly.

import networkx as nx

def build_nx_graph(points):
    """Build a networkx DiGraph from GPS points for validation."""
    G = nx.DiGraph()
    for i in range(len(points) - 1):
        lat1, lon1, e1 = points[i]
        lat2, lon2, e2 = points[i+1]
        d = haversine(lat1, lon1, lat2, lon2)
        G.add_edge(i, i+1, dist=d,
                   elev=max(0, e2-e1),
                   naismith=(d/1000)/5 + max(0,e2-e1)/600)
        G.add_edge(i+1, i, dist=d,
                   elev=max(0, e1-e2),
                   naismith=(d/1000)/5 + max(0,e1-e2)/600)
    return G


G_rila = build_nx_graph(rila_points)

# Compare distance result
nx_dist = nx.shortest_path_length(G_rila, 0, TARGET_RILA, weight='dist')
my_dist = rila_results[1]['cost']

print('Validation — Rila, shortest distance:')
print(f'  Our Dijkstra : {my_dist/1000:.4f} km')
print(f'  networkx     : {nx_dist/1000:.4f} km')
print(f'  Match        : {abs(my_dist - nx_dist) < 1e-6}')
print()

# Compare Naismith result
nx_time = nx.shortest_path_length(G_rila, 0, TARGET_RILA, weight='naismith')
my_time = rila_results[3]['cost']

print('Validation — Rila, Naismith time:')
print(f'  Our Dijkstra : {my_time:.4f} hours')
print(f'  networkx     : {nx_time:.4f} hours')
print(f'  Match        : {abs(my_time - nx_time) < 1e-6}')